<center><h1>Data Cleaning</h1></center>

### Merging all the csv files and converting them to pandas dataframe

##### **Reason:** 

##### Multiple patient CGM information CSV files were consolidated into a single dataset to enable efficient analysis across all patients. Since each file originally represented an individual patient, a new column `patient_id` was added (derived from the CSV file name) to preserve patient identity after merging. Then, the combined dataset was stored as a single sheet to support aggregation, filtering, and time-based analysis. This structure follows a relational format, allowing the dataset to be linked via `patient_id` for advanced analysis while avoiding data redundancy and improving scalability.

In [1]:
import pandas as pd
import glob
import os

# 1. Get file list
path = 'path_to_csv_files'
all_files = glob.glob(os.path.join("HUPA-UC_Diabetes_Dataset\HUPA*.csv"))

# 2. & 3. Read, add columns, and append to list
li = []
for filename in all_files:
    df = pd.read_csv(filename,sep=";")
    patientid = filename.split(".")[0]
    df['Patient_ID'] = os.path.basename(patientid) # Adding column
    li.append(df)

# Concatenate all into one DataFrame
df_final = pd.concat(li, ignore_index=True)

In [2]:
df_final

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2018-06-13T18:40:00,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P
1,2018-06-13T18:45:00,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P
2,2018-06-13T18:50:00,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P
3,2018-06-13T18:55:00,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P
4,2018-06-13T19:00:00,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P
...,...,...,...,...,...,...,...,...,...
309387,2022-05-18T11:55:00,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P
309388,2022-05-18T12:00:00,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P
309389,2022-05-18T12:05:00,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P
309390,2022-05-18T12:10:00,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P


In [3]:
#df_final.to_excel("output.xlsx", index=False, sheet_name='DataSheet')

### Seperating data and time columns to analyse hourly and date wise data

##### **Reason:***

##### The `time` column is a full timestamp and it can't be directly used in groupby operations or as a feature when deriving insights. Therefore, breaking it into separate columns `time` and `date` helps with all date and time based analysis. If not, we have to separate the column inside every time based analysis.

In [3]:
# Split by the space delimiter and expand into two columns
df_final[['Date', 'Time']] = df_final['time'].astype(str).str.split('T', expand=True)
df_final

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID,Date,Time
0,2018-06-13T18:40:00,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:40:00
1,2018-06-13T18:45:00,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:45:00
2,2018-06-13T18:50:00,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:50:00
3,2018-06-13T18:55:00,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:55:00
4,2018-06-13T19:00:00,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P,2018-06-13,19:00:00
...,...,...,...,...,...,...,...,...,...,...,...
309387,2022-05-18T11:55:00,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,11:55:00
309388,2022-05-18T12:00:00,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:00:00
309389,2022-05-18T12:05:00,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:05:00
309390,2022-05-18T12:10:00,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:10:00


### Dropping old time column from dataframe as we have generated two new columns Date and Time.

In [4]:
# drop old datetime column
df = df_final.drop('time', axis=1)
df
if 'Patient_ID' in df:
    print("exists")

exists


### Merging all patients' data to incude all the columns from T1DM_patient_sleep_demographics_with_race.xlsx

##### **Reason:** 

##### Merging the patients' demographic and sleep data with the main sheet for the ease of faster calculations.

In [6]:
df2 = pd.read_excel('HUPA-UC_Diabetes_Dataset\T1DM_patient_sleep_demographics_with_race.xlsx')

# Join the files on the common 'Patient ID' column
# Use how='inner' to keep only matches, or how='left' to keep all from file1
merged_df = pd.merge(df, df2, on='Patient_ID', how='inner')

# Save the result to a new Excel file
#merged_df.to_excel('merged_patients.xlsx', index=False)

### Dropping duplicate rows, if any

In [7]:
df = merged_df.drop_duplicates()
df

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID,Date,Time,Age,Gender,Race,Average Sleep Duration (hrs),Sleep Quality (1-10),% with Sleep Disturbances
0,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:40:00,34,Male,Other,6.3,4.5,80
1,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:45:00,34,Male,Other,6.3,4.5,80
2,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:50:00,34,Male,Other,6.3,4.5,80
3,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:55:00,34,Male,Other,6.3,4.5,80
4,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P,2018-06-13,19:00:00,34,Male,Other,6.3,4.5,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309387,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,11:55:00,62,Male,Black,5.1,7.9,30
309388,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:00:00,62,Male,Black,5.1,7.9,30
309389,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:05:00,62,Male,Black,5.1,7.9,30
309390,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:10:00,62,Male,Black,5.1,7.9,30


In [8]:
merged_df.drop_duplicates()

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID,Date,Time,Age,Gender,Race,Average Sleep Duration (hrs),Sleep Quality (1-10),% with Sleep Disturbances
0,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:40:00,34,Male,Other,6.3,4.5,80
1,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:45:00,34,Male,Other,6.3,4.5,80
2,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:50:00,34,Male,Other,6.3,4.5,80
3,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:55:00,34,Male,Other,6.3,4.5,80
4,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P,2018-06-13,19:00:00,34,Male,Other,6.3,4.5,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309387,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,11:55:00,62,Male,Black,5.1,7.9,30
309388,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:00:00,62,Male,Black,5.1,7.9,30
309389,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:05:00,62,Male,Black,5.1,7.9,30
309390,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:10:00,62,Male,Black,5.1,7.9,30


In [9]:
df

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID,Date,Time,Age,Gender,Race,Average Sleep Duration (hrs),Sleep Quality (1-10),% with Sleep Disturbances
0,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:40:00,34,Male,Other,6.3,4.5,80
1,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:45:00,34,Male,Other,6.3,4.5,80
2,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:50:00,34,Male,Other,6.3,4.5,80
3,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:55:00,34,Male,Other,6.3,4.5,80
4,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P,2018-06-13,19:00:00,34,Male,Other,6.3,4.5,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309387,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,11:55:00,62,Male,Black,5.1,7.9,30
309388,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:00:00,62,Male,Black,5.1,7.9,30
309389,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:05:00,62,Male,Black,5.1,7.9,30
309390,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:10:00,62,Male,Black,5.1,7.9,30


### Checking for NULL values in dataframe

In [10]:
df.isnull().values.any()

np.False_

### Checking for NULL values in glucose column

##### **Reason:**

##### Checking for NULL values in the glucose column is an important data cleaning step because glucose measurements are one of the primary variables used to analyze diabetes management and patient health. Missing glucose values can affect the accuracy of statistical analysis and visualizations.

In [11]:
df['glucose'].isnull().any()

np.False_

### Standardizing column names - Converting all column names to lower case and replace blank spaces with underscore(_)

##### **Reason:**

##### Converting all column names to lower case and replace " " spaces with underscore (_) to make it python readable programming format

In [12]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id,date,time,age,gender,race,average_sleep_duration_(hrs),sleep_quality_(1-10),%_with_sleep_disturbances
0,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:40:00,34,Male,Other,6.3,4.5,80
1,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:45:00,34,Male,Other,6.3,4.5,80
2,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:50:00,34,Male,Other,6.3,4.5,80
3,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P,2018-06-13,18:55:00,34,Male,Other,6.3,4.5,80
4,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P,2018-06-13,19:00:00,34,Male,Other,6.3,4.5,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309387,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,11:55:00,62,Male,Black,5.1,7.9,30
309388,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:00:00,62,Male,Black,5.1,7.9,30
309389,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:05:00,62,Male,Black,5.1,7.9,30
309390,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P,2022-05-18,12:10:00,62,Male,Black,5.1,7.9,30


In [13]:
df['%_with_sleep_disturbances']

0         80
1         80
2         80
3         80
4         80
          ..
309387    30
309388    30
309389    30
309390    30
309391    30
Name: %_with_sleep_disturbances, Length: 309392, dtype: int64

In [ ]:
df.to_excel("cleaned_data.xlsx", index=False, sheet_name='DataSheet')

### Handling negative `bolus_volume_delivered`

##### **Reason**: 

##### `bolus_volume_delivered` contains values of -1 and -3 in HUPA0017P. Insulin delivery volume is a physical quantity and cannot be negative. These values could be incorrect reading from device. We replaced all negative bolus values with 0 which means that no insulin was delivered. If left uncleaned, negative bolus values would lower the average bolus dose and hinder any analysis of insulin-glucose response.

In [17]:
# Reading file
#df_all = pd.read_excel("cleaned_data.xlsx")

# Checking negative bolus values per patient
negative_bolus = df[df['bolus_volume_delivered'] < 0]

print(f"Total negative bolus readings: {len(negative_bolus)}")

print(negative_bolus.groupby('patient_id')['bolus_volume_delivered'].value_counts())

# Cleaning bolus values
def clean_bolus(group):
    patient_id = group.name
    
    # Count bad readings
    bad_bolus = group['bolus_volume_delivered'] < 0
    
    print(f"{patient_id}: {bad_bolus.sum()} negative bolus readings found")
    
    # Replace negative values with 0
    group.loc[bad_bolus, 'bolus_volume_delivered'] = 0
    
    return group

# Apply cleaning
cols = df.columns

df = (df.groupby('patient_id', group_keys=False)[cols].apply(clean_bolus).reset_index(drop=True))

print("Bolus cleaning complete")

print(
    f"Remaining negative values: "
    f"{(df['bolus_volume_delivered'] < 0).sum()}"
)

Total negative bolus readings: 4
patient_id  bolus_volume_delivered
HUPA0017P   -1.0                      3
            -3.0                      1
Name: count, dtype: int64
HUPA0001P: 0 negative bolus readings found
HUPA0002P: 0 negative bolus readings found
HUPA0003P: 0 negative bolus readings found
HUPA0004P: 0 negative bolus readings found
HUPA0005P: 0 negative bolus readings found
HUPA0006P: 0 negative bolus readings found
HUPA0007P: 0 negative bolus readings found
HUPA0009P: 0 negative bolus readings found
HUPA0010P: 0 negative bolus readings found
HUPA0011P: 0 negative bolus readings found
HUPA0014P: 0 negative bolus readings found
HUPA0015P: 0 negative bolus readings found
HUPA0016P: 0 negative bolus readings found
HUPA0017P: 4 negative bolus readings found
HUPA0018P: 0 negative bolus readings found
HUPA0019P: 0 negative bolus readings found
HUPA0020P: 0 negative bolus readings found
HUPA0021P: 0 negative bolus readings found
HUPA0022P: 0 negative bolus readings found
HUPA0023P

### Rounding columns `glucose`, `calories` and `heart_rate` to 2 decimal places and `basal_rate` to 3 decimal places

##### **Reason:**

##### Rounding columns `glucose`, `calories`, `heart_rate` and `basal_rate` for the ease of readability for insights and visualizations.

In [18]:
# Rounding the columns to 2 decimal places
df = df.round({'glucose': 2, 'calories': 2, 'heart_rate': 2, 'basal_rate': 3})

print("Change complete")

Change complete


In [19]:
df.to_excel("Team6_DataDynamos_Python-Hackathon_MAY2026_V2.xlsx", index=False, sheet_name='DataSheet')

print("New clean data file that contains handling of bolus_volume_delivered negative values + rounding some columns to 2 and 3 decimal places")

New clean data file that contains handling of bolus_volume_delivered negative values + rounding some columns to 2 and 3 decimal places
